# Lip-sync your character on a free GPU

1. Menu **Runtime > Change runtime type > T4 GPU**, then **Save**.
2. Press **Runtime > Run all**.
3. When asked, upload your **Midjourney clip** (.mp4) and your **voice sample** (10-20 s .wav of the voice to copy, used with permission).
4. Type the line she should say in step 3.
5. The finished video downloads at the end.

First run takes about 10 minutes to install. Your files stay in your own Colab session.

In [ ]:
# 1) check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) install LivePortrait (face/mouth animation) and Chatterbox (voice copy)
%cd /content
!git clone -q --depth 1 https://github.com/KwaiVGI/LivePortrait.git
!git clone -q --depth 1 -b claude/python-midjourney-video-lib-fc303z https://github.com/modernpinkpaper/Test.git mytools
!pip install -q chatterbox-tts "setuptools<81" 2>&1 | tail -2
!pip install -q onnxruntime-gpu pyyaml opencv-python-headless scipy imageio imageio-ffmpeg tqdm rich ffmpeg-python onnx scikit-image albumentations tyro pykalman soundfile num2words 2>&1 | tail -2
from huggingface_hub import snapshot_download
snapshot_download('KwaiVGI/LivePortrait', local_dir='/content/LivePortrait/pretrained_weights', ignore_patterns=['*.git*', 'README.md', 'docs/*'])
print('installed')

In [ ]:
# 3) the line she says
LINE = "Hey mama! You take care of everyone all day. Tonight, drink your water and take care of you."
START = 1.0      # seconds into the clip when she starts talking
EMOTION = 0.6    # 0.5 = normal, higher = more expressive
SPEED = 1.1      # 1.0 = normal speed

In [ ]:
# 4) upload the clip and the voice sample
from google.colab import files
import shutil
print('Upload your Midjourney clip (.mp4)')
clip = list(files.upload().keys())[0]
print('Upload your voice sample (.wav)')
ref = list(files.upload().keys())[0]
shutil.move(clip, '/content/clip.mp4'); shutil.move(ref, '/content/voice_ref.wav')

In [ ]:
# 5) make the voice line
import re, subprocess, numpy as np, soundfile as sf, torch, torchaudio
from num2words import num2words
from chatterbox.tts import ChatterboxTTS
tts = ChatterboxTTS.from_pretrained(device='cuda')
text = re.sub(r'\d+', lambda m: num2words(int(m.group())), LINE)
w = tts.generate(text, audio_prompt_path='/content/voice_ref.wav', exaggeration=EMOTION, cfg_weight=0.5)
w = torchaudio.functional.resample(w, tts.sr, 24000).squeeze(0).cpu().numpy()
loud = np.where(np.abs(w) > 0.01)[0]; w = w[max(loud[0]-960,0):loud[-1]+960]
sf.write('/content/line_raw.wav', w, 24000)
subprocess.run(['ffmpeg','-loglevel','error','-y','-i','/content/line_raw.wav','-filter:a',f'atempo={SPEED}','/content/line.wav'], check=True)
del tts; torch.cuda.empty_cache()
from IPython.display import Audio; Audio('/content/line.wav')

In [ ]:
# 6) lip-sync her to the voice
!python /content/mytools/demo_videos/lipsync_liveportrait.py --lp /content/LivePortrait /content/clip.mp4 /content/line.wav /content/lipsync.mp4 --start {START}

In [ ]:
# 7) watch and download
!ffmpeg -loglevel error -y -i /content/lipsync.mp4 -c:v libx264 -crf 20 -c:a copy /content/lipsync_final.mp4
from IPython.display import Video
files.download('/content/lipsync_final.mp4')
Video('/content/lipsync_final.mp4', embed=True, width=360)